# Case Study No. 124: Content Popularity Analysis
## Project Title: Supervised Machine Learning System for Digital Content Engagement Forecasting

### Objective:
To understand, analyze, and estimate which online articles are likely to attract high reader engagement (shares) using structured metadata, content characteristics, and machine learning classification algorithms.

### 1. Environment Setup & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib

sns.set_theme(style='whitegrid')
print('Libraries imported successfully!')

### 2. Dataset Loading & Inspection
We utilize the **UCI Online News Popularity Dataset** consisting of 39,644 articles from Mashable over a 2-year acquisition period.

In [ ]:
df = pd.read_csv('data/OnlineNewsPopularity.csv')
df.columns = df.columns.str.strip()
print(f'Dataset Shape: {df.shape}')
df.head()

### 3. Target Formulation (Problem Definition)
Engagement is measured by social shares. Due to the heavy right skew of virality, we formulate a binary classification problem around the median value (1,400 shares):
* **Class 1 (High Engagement / Popular)**: $\text{shares} \ge 1,400$
* **Class 0 (Standard / Low Engagement)**: $\text{shares} < 1,400$

In [ ]:
median_shares = df['shares'].median()
print(f'Median Shares: {median_shares}')
df['is_popular'] = (df['shares'] >= median_shares).astype(int)
print('Class balance:\n', df['is_popular'].value_counts(normalize=True))

### 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Engagement by Channel
channels = ['lifestyle', 'entertainment', 'bus', 'socmed', 'tech', 'world']
ch_rates = {}
for ch in channels:
    col = f'data_channel_is_{ch}'
    ch_rates[ch.capitalize()] = df[df[col] == 1]['is_popular'].mean() * 100

plt.figure(figsize=(8, 4))
sns.barplot(x=list(ch_rates.keys()), y=list(ch_rates.values()), palette='viridis')
plt.axhline(53.4, color='r', linestyle='--', label='Average Rate (53.4%)')
plt.ylabel('% High Engagement')
plt.title('High Engagement Rate across Content Channels')
plt.legend()
plt.show()

In [ ]:
# 4.2 Weekend vs Weekday Engagement
plt.figure(figsize=(6, 4))
sns.barplot(x='is_weekend', y='is_popular', data=df, palette='Blues_d')
plt.xticks([0, 1], ['Weekday (Mon-Fri)', 'Weekend (Sat-Sun)'])
plt.ylabel('Proportion Popular')
plt.title('Weekend vs Weekday Popularity Advantage')
plt.show()

### 5. Feature Selection & Preprocessing
We select 17 intuitive and interpretable features spanning content structure, multimedia, channels, timing, and sentiment.

In [ ]:
feature_cols = [
    'n_tokens_title', 'n_tokens_content', 'num_hrefs', 'num_imgs',
    'num_videos', 'num_keywords', 'data_channel_is_lifestyle',
    'data_channel_is_entertainment', 'data_channel_is_bus',
    'data_channel_is_socmed', 'data_channel_is_tech', 'data_channel_is_world',
    'is_weekend', 'global_sentiment_polarity', 'title_sentiment_polarity',
    'title_subjectivity', 'kw_avg_avg'
]

X = df[feature_cols].fillna(0)
y = df['is_popular']

# Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples')

### 6. Model Training & Comparison
We compare three foundational machine learning algorithms:
1. **Logistic Regression** (Linear baseline, high interpretability)
2. **Decision Tree** (Non-linear rule-based tree)
3. **Random Forest** (Ensemble bagging of decision trees)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10, random_state=42, n_jobs=-1)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc * 100:.2f}%',
        'Precision': f'{prec * 100:.2f}%',
        'Recall': f'{rec * 100:.2f}%',
        'F1-Score': f'{f1 * 100:.2f}%',
        'ROC-AUC': f'{auc:.4f}'
    })

pd.DataFrame(results)

### 7. Evaluation & Confusion Matrix Analysis

In [ ]:
best_rf = models['Random Forest']
y_pred_rf = best_rf.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Low', 'Pred High'],
            yticklabels=['Actual Low', 'Actual High'])
plt.title('Random Forest Confusion Matrix')
plt.show()

print(classification_report(y_test, y_pred_rf, target_names=['Low (<1.4k)', 'High (>=1.4k)']))

### 8. Feature Importance Analysis

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances.values, y=importances.index, palette='mako')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.show()

### 9. Summary & Findings
1. **Random Forest** achieved the best overall performance with **65.22% Accuracy** and **0.7088 ROC-AUC**.
2. **Top Drivers of Popularity**: Keyword popularity index (`kw_avg_avg`), content length, multimedia elements (images/videos), and weekend publication schedule.
3. The trained models and scalers are exported to `models/` for serving in the Streamlit application.